# Actividad 04: Esquema Estrella y Arquitectura del Data Warehouse

En esta actividad vas a diseñar un **esquema estrella** usando el enfoque de Kimball y luego lo implementarás a través de las **capas del Data Warehouse**: Source, Staging y Storage.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect(':memory:')
print('Base de datos DuckDB en memoria creada.')

## Qué aprenderás

1. Aplicar los **4 pasos de Kimball** para diseñar un esquema estrella
2. Declarar el **grain** del modelo
3. Identificar **dimensiones** y **hechos**
4. Mover datos a través de las **capas del DW**

### El camino de Kimball

Primero el proceso y el grain, después las dimensiones y los hechos: así llegas al **esquema estrella**.

```mermaid
flowchart LR
    P["1. Proceso de negocio"] --> G["2. Grain"]
    G --> D["3. Dimensiones"]
    D --> F["4. Hechos y medidas"]
    F --> S["Esquema estrella"]
```


---

## El caso de negocio

Las preguntas de negocio que debe responder el warehouse están en el [brief de Wesker](../BRIEF.md).

Necesitas un **esquema estrella** con tablas de hechos y dimensiones capaz de responderlas.

---

## Paso 1: Identifica el proceso de negocio

> **Recuerda:** el proceso de negocio es el flujo operativo que generamos como hechos.

El proceso principal es: **experimentos realizados**.

Cada fila en la tabla de hechos representará **un experimento**.

---

## Paso 2: Declara el grain (nivel de detalle)

> **Recuerda:** el **grain** define qué representa exactamente una fila de la tabla de hechos: cuánto más fino, más preguntas responde.

El grain es: **un experimento por fila**.

Cada fila en `experiment_fact` representa un experimento único.

---

## Paso 3: Identifica las dimensiones

> **Recuerda:** las **dimensiones** aportan el contexto de los hechos (quién, qué, cuándo, dónde); los **hechos** son las medidas numéricas.

| Dimensión | Responde a | Columnas clave |
|-----------|------------|----------------|
| `date_dim` | **¿Cuándo?** | date_id, full_date, year, month, is_weekend |
| `laboratory_dim` | **¿Dónde?** | laboratory_id, laboratory_name, country |
| `researcher_dim` | **¿Quién?** | researcher_id, full_name, department |
| `sample_dim` | **¿Qué?** | sample_id, sample_type, risk_level |
| `project_dim` | **¿Qué proyecto?** | project_id, project_name, category |

---

## Paso 4: Identifica los hechos (medidas)

| Hecho | Tipo | Descripción |
|-------|------|-------------|
| `experiment_cost` | Aditivo | Coste total del experimento |
| `processing_time_minutes` | Aditivo | Tiempo de procesamiento |

### El esquema estrella que estás diseñando

La tabla de hechos en el centro y las dimensiones alrededor: cada fila de `experiment_fact` es **un experimento**, y las dimensiones responden a quién, dónde, cuándo y qué.

```mermaid
erDiagram
    DATE_DIM ||--o{ EXPERIMENT_FACT : FK-date
    LABORATORY_DIM ||--o{ EXPERIMENT_FACT : FK-lab
    RESEARCHER_DIM ||--o{ EXPERIMENT_FACT : FK-researcher
    SAMPLE_DIM ||--o{ EXPERIMENT_FACT : FK-sample
    PROJECT_DIM ||--o{ EXPERIMENT_FACT : FK-project
    EXPERIMENT_FACT {
        int experiment_id PK
        int date_id FK
        int laboratory_id FK
        int researcher_id FK
        int sample_id FK
        int project_id FK
        float experiment_cost
        int processing_time_minutes
    }
    DATE_DIM {
        int date_id PK
        date full_date
        int year
        int month
        bool is_weekend
    }
    LABORATORY_DIM {
        int laboratory_id PK
        string laboratory_name
        string country
    }
    RESEARCHER_DIM {
        int researcher_id PK
        string full_name
        string department
    }
    SAMPLE_DIM {
        int sample_id PK
        string sample_type
        string risk_level
    }
    PROJECT_DIM {
        int project_id PK
        string project_name
        string category
    }
```


---

## Arquitectura del DW: las 4 capas

> **Recuerda:** el DW se organiza en 4 capas: **Source** (datos crudos), **Staging** (extracción y limpieza), **Storage** (DW analítico) y **Presentation** (exposición a los usuarios).

En esta actividad movemos datos por las 3 primeras:

```
Source (CSV) -> Staging (tablas limpias) -> Storage (DW estrella)
```

| Capa | Qué hace | Ejemplo |
|------|----------|---------|
| **Source** | Datos crudos sin procesar | Archivos CSV |
| **Staging** | Limpieza y transformación | Tablas DuckDB |
| **Storage** | DW analítico | Esquema estrella |
| **Presentation** | Exposición a los usuarios | Las consultas analíticas de la actividad 05 |

El flujo es: **Source -> Staging -> Storage**; la capa **Presentation** la tocarás en la actividad 05.

### El flujo entre capas

En esta actividad recorres las 3 primeras capas. La cuarta (**Presentation**) es la que usarás para responder preguntas en la actividad 05.

```mermaid
flowchart LR
    SRC["Source<br/>CSV crudos"] --> STG["Staging<br/>tablas limpias"]
    STG --> STO["Storage<br/>DW estrella"]
    STO --> PRE["Presentation<br/>consultas de la actividad 05"]
```


---

## Parte 1: Source (datos crudos)

La carpeta `source/` contiene los datos crudos de Parasol Corp:
- `experimentos_raw.csv`: datos de experimentos
- `investigadores_raw.csv`: datos de investigadores
- `laboratorios_raw.csv`: datos de laboratorios

### Tarea 1.1: Explora los datos crudos

Lee los archivos CSV y muestra las primeras filas de cada uno.

In [ ]:
# TODO: Lee los 3 archivos CSV y muestra las primeras filas
# Usa pd.read_csv('source/experimentos_raw.csv')
# Pista: df.head() muestra las primeras filas


### Tarea 1.2: Identifica problemas en los datos

Los datos crudos pueden tener problemas. Observa:
- Valores nulos
- Tipos de datos incorrectos
- Datos inconsistentes

In [ ]:
# TODO: Muestra información de cada DataFrame
# Pista: df.info() muestra tipos y nulos


---

## Parte 2: Staging (limpieza y transformación)

La carpeta `staging/` es donde cargamos los datos crudos a tablas de DuckDB y los limpiamos.

### Tarea 2.1: Carga los datos a tablas de staging

Crea tablas en DuckDB desde los CSV.

In [ ]:
# TODO: Crea la tabla staging_experimentos desde el CSV
# con.execute("CREATE TABLE staging_experimentos AS SELECT * FROM read_csv_auto('source/experimentos_raw.csv')")


In [ ]:
# TODO: Crea la tabla staging_investigadores desde el CSV


In [ ]:
# TODO: Crea la tabla staging_laboratorios desde el CSV


### Tarea 2.2: Limpia los datos

Verifica que los datos están limpios:
- Sin valores nulos en columnas críticas
- Tipos de datos correctos

In [ ]:
# TODO: Verifica que no hay nulos en columnas críticas
# SELECT COUNT(*) FROM staging_experimentos WHERE experiment_id IS NULL


---

## Parte 3: Storage (Data Warehouse)

La carpeta `storage/` es donde vive el Data Warehouse con esquema estrella.

### Tarea 3.1: Crea las dimensiones del DW

Recuerda las dimensiones que identificamos en el Paso 3:
- `laboratory_dim`: donde ocurrio
- `researcher_dim`: quien lo realizo
- `date_dim`: cuando ocurrio

In [ ]:
# TODO: Crea la tabla laboratory_dim desde staging_laboratorios
# Debe tener: laboratory_id, laboratory_name, country


In [ ]:
# TODO: Crea la tabla researcher_dim desde staging_investigadores
# Debe tener: researcher_id, full_name, department


In [ ]:
# TODO: Crea la tabla date_dim con fechas únicas de los experimentos
# Debe tener: date_id, full_date, year, month


### Tarea 3.2: Crea la tabla de hechos

La tabla `experiment_fact` conecta todas las dimensiones.

Recuerda: cada fila representa **un experimento** (el grain que declaramos en el Paso 2).

In [ ]:
# TODO: Crea la tabla experiment_fact con FKs a las dimensiones
# Debe tener: experiment_id, laboratory_id, researcher_id, date_id, experiment_cost


### Tarea 3.3: Puebla el DW desde staging

Inserta los datos de staging a las tablas del DW.

### De staging al DW

Las tablas de staging alimentan a las dimensiones y la tabla de hechos se rellena cruzando staging con las dimensiones ya construidas.

```mermaid
flowchart TB
    SE[staging_experimentos] --> EF[experiment_fact]
    SE --> DD[date_dim]
    SL[staging_laboratorios] --> LD[laboratory_dim]
    SI[staging_investigadores] --> RD[researcher_dim]
    LD --> EF
    RD --> EF
    DD --> EF
```


In [ ]:
# TODO: Puebla laboratory_dim desde staging_laboratorios


In [ ]:
# TODO: Puebla researcher_dim desde staging_investigadores


In [ ]:
# TODO: Puebla experiment_fact desde staging_experimentos


### Tarea 3.4: Consulta analítica

Verifica que tu DW responde a las preguntas de Wesker.

```sql
-- Cuanto cuesta cada laboratorio?
SELECT l.laboratory_name, SUM(f.experiment_cost) as total_cost
FROM experiment_fact f
JOIN laboratory_dim l ON f.laboratory_id = l.laboratory_id
GROUP BY l.laboratory_name;
```

In [ ]:
# TODO: Escribe un JOIN que muestre laboratory_name y SUM(experiment_cost)


---

## Ampliación: el esquema snowflake

Tu estrella mantiene las dimensiones **desnormalizadas**: por ejemplo
`laboratory_dim` guarda el país dentro de la propia dimensión. Si **normalizas
una dimensión**, sacando sus atributos a tablas aparte, el esquema deja de ser
una estrella y pasa a ser un esquema **snowflake**: las dimensiones se abren en
ramas, como un copo de nieve.

Separando el país de `laboratory_dim` en una nueva `country_dim`:

```mermaid
erDiagram
    COUNTRY_DIM ||--o{ LABORATORY_DIM : FK-country
    LABORATORY_DIM ||--o{ EXPERIMENT_FACT : FK-lab
    COUNTRY_DIM {
        int country_id PK
        string country
    }
    LABORATORY_DIM {
        int laboratory_id PK
        string laboratory_name
        int country_id FK
    }
    EXPERIMENT_FACT {
        int experiment_id PK
        int laboratory_id FK
        float experiment_cost
    }
```

La única diferencia está en la dimensión: `LABORATORY_DIM` ya no guarda el país,
sino una `country_id` que apunta a la tabla nueva.

| | Estrella | Snowflake |
|---|---|---|
| Dimensiones | desnormalizadas | normalizadas |
| Consultas | menos JOINs | más JOINs |
| Espacio | algo de redundancia | menos redundancia |

En la práctica casi todos los DW usan estrella: la redundancia de las dimensiones
es pequeña y las consultas van más rápidas. Pero conviene reconocer un snowflake
cuando te lo encuentres.


---

## Resumen

En esta actividad has:

1. Aplicado los **4 pasos de Kimball** (proceso, grain, dimensiones, hechos).
2. Movido datos por las capas **Source -> Staging -> Storage**.
3. Construido el esquema estrella de Parasol Corp.

## Guarda tu trabajo

Copia tu solución en `scripts/star_schema_layers.py`.

```bash
pytest tests/ -v
```

### Siguiente: Actividad 05: Consultas analíticas